<a href="https://colab.research.google.com/github/mikakia/AdvanceML_Final_Project/blob/main/original_unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# Functions

In [ ]:
# for downdsamplig
def double_conv(input_c, output_c):
  conv = nn.Sequential(
                                 nn.Conv2d(in_channels=input_c,out_channels=output_c, kernel_size=3),      # (2)
                                 nn.ReLU(inplace=True) ,                                                   # (3)
                                 nn.Conv2d(in_channels=output_c,out_channels=output_c, kernel_size=3),     # second unpadded convolution
                                 nn.ReLU(inplace=True)  )
  return conv

def double_conv(input_c, output_c):
  conv = nn.Sequential(
                                 nn.Conv2d(in_channels=input_c,out_channels=output_c, kernel_size=3),      # (2)
                                 nn.ReLU(inplace=True) ,                                                   # (3)
                                 nn.Conv2d(in_channels=output_c,out_channels=output_c, kernel_size=3),     # second unpadded convolution
                                 nn.ReLU(inplace=True)  )
  return conv

# function to use to crop images in order to be the same size so they can be concatenated
def crop_image(tensor, target_sensor):
  target_size = target_sensor.size()[2]   # [2] because we have square images
  tensor_size = tensor.size()[2]
  if tensor_size > target_size:
    delta = tensor_size -  target_size
    delta = delta//2
    return tensor[:,:,delta:tensor_size-delta,delta:tensor_size-delta]
  else:
    print("The input image size is bigger than then output image")
    exit(0)

# Defining UNet Class

Building a UNet based on the orginal UNet paper.

*   The contracting part (left side) follows the typical architecture of a convulotinal neural network.
*   It consists of the repeated application of two 3x3 unpadded convolutions(2), each followed by ReLU (3)(4)and a 2x2 max pooling operation with stride 2 for downsampling(1). At each downsampling step we double the number of feature channels (5).
*   Upsampling and transpose convolutions (6)
*   Upsampling of the feature map followed by a 2x2 convolution that halves the
number of feature channels, a concatenation with the correspondingly cropped
feature map from the contracting path, and two 3x3 convolutions, each fol-
lowed by a ReLU.






In [28]:
class UNet(nn.Module):

  def __init__(self):
    super(UNet, self).__init__()                                   # initiliazion for nn.Module
    self.max_pool_2x2 = nn.MaxPool2d(kernel_size=2,stride=2)       # (1)

    # first part Downsampling (5)
    self.down_conv_1 = double_conv(1,64)
    self.down_conv_2 = double_conv(64,128)
    self.down_conv_3 = double_conv(128,256)
    self.down_conv_4 = double_conv(256,512)
    self.down_conv_5 = double_conv(512,1024)

    #  second part Upsampling (6) transpose convolution

    # First
    self.up_trans_1 = nn.ConvTranspose2d(in_channels=1024,          # input channels from the last down_conv5 and
                                        out_channels=512,           # output channels from the output of down_conv4 which is passed to the decoder
                                        kernel_size=2,stride=2)

    self.up_conv_1 = double_conv(1024,512)


    # Second
    self.up_trans_2 = nn.ConvTranspose2d(in_channels=512,          # input channels from the last down_conv5 and
                                        out_channels=256,          # output channels from the output of down_conv4 which is passed to the decoder
                                        kernel_size=2,stride=2)

    self.up_conv_2 = double_conv(512,256)


    # Third
    self.up_trans_3 = nn.ConvTranspose2d(in_channels=256,          # input channels from the last down_conv5 and
                                        out_channels=128,           # output channels from the output of down_conv4 which is passed to the decoder
                                        kernel_size=2,stride=2)

    self.up_conv_3= double_conv(256,128)

    # Fourth
    self.up_trans_4 = nn.ConvTranspose2d(in_channels=128,          # input channels from the last down_conv5 and
                                        out_channels=64,           # output channels from the output of down_conv4 which is passed to the decoder
                                       kernel_size=2,stride=2)

    self.up_conv_4 = double_conv(128,64)

    # Output layer
    self.output = nn.Conv2d(in_channels=64,out_channels=2, kernel_size=1)  # 2d convolition with kernel size of 1

  def forward(self,image):

    # batch size, channels, height and width
    print("\n =============ENCODER===============\n")
    # encoder layers
    x1 = self.down_conv_1(image) # output passed to the decoder
    print("First encoder layer:", x1.size())
    x2 = self.max_pool_2x2(x1)
    x3 = self.down_conv_2(x2)    # output passed to the decoder
    x4 = self.max_pool_2x2(x3)
    x5 = self.down_conv_3(x4)    # output passed to the decoder
    x6 = self.max_pool_2x2(x5)
    x7 = self.down_conv_4(x6)    # output passed to the decoder
    print("4th encoder layer size x7", x7.size())
    x8 = self.max_pool_2x2(x7)
    x9 = self.down_conv_5(x8)
    print("Last encoder layer x9:",x9.size())

    # decoder layers

    print("\n =============DECODER===============\n")
    # First
    x = self.up_trans_1(x9)                       # x should be concatened with x7 to create the skip connections in Unet
    print("First decoder layer x: ",x.size())
    y = crop_image(x7,x)
    print("Size of concatenated image y : ",y.size())
    x = self.up_conv_1(torch.cat([x,y],1))
    print(x.size())

    # Second

    x = self.up_trans_2(x)                       # x should be concatened with x7 to create the skip connections in Unet
    print("Second decoder layer x: ",x.size())
    y = crop_image(x5,x)
    print("Size of concatenated image y : ",y.size())
    x = self.up_conv_2(torch.cat([x,y],1))
    print(x.size())


    # Third

    x = self.up_trans_3(x)                       # x should be concatened with x7 to create the skip connections in Unet
    print("Third decoder layer x: ",x.size())
    y = crop_image(x3,x)
    print("Size of concatenated image y : ",y.size())
    x = self.up_conv_3(torch.cat([x,y],1))
    print(x.size())




    # Fourth

    x = self.up_trans_4(x)                       # x should be concatened with x7 to create the skip connections in Unet
    print("Fourth decoder layer x: ",x.size())
    y = crop_image(x1,x)
    print("Size of concatenated image y : ",y.size())
    x = self.up_conv_4(torch.cat([x,y],1))
    print(x.size())

    # Output
    out = self.output(x)
    print("Output size: ",out.size())

    return out

In [29]:
if __name__ == "__main__":
    image = torch.rand((1,1,572,572)) # batch size=1, channels=1, height=572 and width=572 following the original paper
    model = UNet()
    print("Model:",model(image))




 =============ENCODER===============

First encoder layer: torch.Size([1, 64, 568, 568])
4th encoder layer size x7 torch.Size([1, 512, 64, 64])
Last encoder layer x9: torch.Size([1, 1024, 28, 28])

 =============DECODER===============

First decoder layer x:  torch.Size([1, 512, 56, 56])
Size of concatenated image y :  torch.Size([1, 512, 56, 56])
torch.Size([1, 512, 52, 52])
Second decoder layer x:  torch.Size([1, 256, 104, 104])
Size of concatenated image y :  torch.Size([1, 256, 104, 104])
torch.Size([1, 256, 100, 100])
Third decoder layer x:  torch.Size([1, 128, 200, 200])
Size of concatenated image y :  torch.Size([1, 128, 200, 200])
torch.Size([1, 128, 196, 196])
Fourth decoder layer x:  torch.Size([1, 64, 392, 392])
Size of concatenated image y :  torch.Size([1, 64, 392, 392])
torch.Size([1, 64, 388, 388])
Output size:  torch.Size([1, 2, 388, 388])
Model: tensor([[[[ 0.0063,  0.0077,  0.0083,  ...,  0.0065,  0.0084,  0.0067],
          [ 0.0074,  0.0090,  0.0106,  ...,  0.0116,